### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [5]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [6]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [7]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [10]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [5, 10]
}

In [11]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/rishabh/ML/Projects/Churn Prediction Using ANN/avenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/rishabh/ML/Projects/Churn Prediction Using ANN/avenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/rishabh/ML/Projects/Churn Prediction Using ANN/avenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/rishabh/ML/Projects/Churn Prediction U

Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5
Epoch 1/5


2026-03-01 14:36:31.365142: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.390317: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.456508: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.466851: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.479176: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.486505: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-03-01 14:36:31.486726: I tensorflow/core/grappler/optimizers/cust

167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.6580 - loss: 0.6476
165/167 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6658 - loss: 0.6163Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7327 - loss: 0.5359
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7238 - loss: 0.5667
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7632 - loss: 0.5178
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7480 - loss: 0.5220
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7739 - loss: 0.4919
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.6704 - loss: 0.6177
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7495 - loss: 0.5250
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.7429 - loss: 0.5412
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6588 - loss: 0.6892
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.8129 - lo

...

167/167 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7709 - loss: 0.4910
Epoch 2/10
 13/167 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.7813 - loss: 0.4544Epoch 1/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.8071 - loss: 0.4340
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.8041 - loss: 0.4537
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/steptep - accuracy: 0.8022 - loss: 0.454
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/steptep - accuracy: 0.8109 - loss: 0.448
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.8076 - loss: 0.4608
Epoch 6/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8125 - loss: 0.4368
Epoch 7/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8054 - loss: 0.4471
Epoch 3/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8071 - loss: 0.4445
Epoch 7/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8061 - loss: 0.4590
Epoch 7/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8058 - loss: 0.4392
Epoch 3/10
16

/Users/rishabh/ML/Projects/Churn Prediction Using ANN/avenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/rishabh/ML/Projects/Churn Prediction Using ANN/avenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-01 14:39:23.770904: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-01 14:39:23.771690: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-01 14:39:23.771716: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-01 14:39:23.771734: I t

Epoch 1/5


2026-03-01 14:39:24.608134: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7591 - loss: 0.5149
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8074 - loss: 0.4355
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8100 - loss: 0.4339
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8095 - loss: 0.4335
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8084 - loss: 0.4343
Best: 0.812501 using {'epochs': 5, 'layers': 1, 'neurons': 32}
